# Week 8 — Final Integration and Validation

Validate the standalone dual-pricing package, quantify the empirical error band, reproduce required stress scenarios and create the final dashboard assets.

## 1. Load the standalone pricing package

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WEEK_DIR = Path.cwd()
TOOL_DIR = WEEK_DIR / "pricing_tool"
RESULTS_DIR = WEEK_DIR / "model_results"
FIGURES_DIR = WEEK_DIR / "figures"
for directory in [RESULTS_DIR, FIGURES_DIR]:
    directory.mkdir(exist_ok=True)
sys.path.insert(0, str(TOOL_DIR))

from pricing_engine import ERROR_INTERVAL, FEATURE_CONTEXT, METADATA, price_contract
from data_update import load_cached_snapshot

snapshot = load_cached_snapshot()
snapshot

Out[1]: 
{'as_of_date': '2024-12-30',
 'retrieved_at_utc': '2026-09-07T09:57:43.434790+00:00',
 'mode': 'offline_week2_fallback',
 'jpm_close': 231.0775604248047,
 'vix_close': 17.399999618530273,
 'risk_free_rate': 0.0455,
 'historical_volatility_20d': 0.1906247800741619,
 'sources': {'JPM_and_VIX': 'G:\\JPM-Chooser Option Pricing\\Week2\\processed_data\\market_data_processed.csv',
  'risk_free_rate': 'G:\\JPM-Chooser Option Pricing\\Week2\\processed_data\\market_data_processed.csv'},
 'update_status': 'cached_fallback',
 'online_error': None,
 'feature_context': {'feature_as_of_date': '2024-12-30',
  'pricing_rate': 0.0455,
  'values': {'Daily_Return': -0.007671073530268413,
   'Abs_Return_1D': 0.007671073530268413,
   'Rolling_Volatility_5D': 0.15949756918498206,
   'Rolling_Volatility_10D': 0.24354147196090928,
   'Rolling_Volatility_20D': 0.19062478007416173,
   'Rolling_Volatility_60D': 0.3057059097260045,
   'VIX_Close': 17.399999618530273,
   'VIX_Return': 0.0909090800379102,
 

## 2. Dual-pricing example

In [2]:
example = price_contract(
    spot=snapshot["jpm_close"], strike=150.0, rate=snapshot["risk_free_rate"],
    dividend_yield=0.0233, volatility=snapshot["historical_volatility_20d"],
    choice_time=0.5, maturity=1.0, vix=snapshot["vix_close"]
)
(RESULTS_DIR / "dual_pricing_example.json").write_text(json.dumps(example, indent=2), encoding="utf-8")
example

Out[2]: 
{'bsm_price': 82.52782127278121,
 'greeks': {'delta': 0.9702147310233546,
  'gamma': 0.00044477615251363277,
  'vega_per_1pct': 0.043461846470904675,
  'rho_per_1pct': 1.416729480724399},
 'ml_available': True,
 'feature_context_date': '2024-12-30',
 'out_of_training_range': {'Close': 231.0775604248047},
 'warning': 'All ML features share the displayed feature date. Input changes are counterfactual scenarios. DGS10 remains the trained ML rate feature; DGS1 supplies the online BSM discount rate.',
 'ml_predicted_forward_volatility': 0.16653673756974308,
 'approach1_ml_vol_bsm_price': 82.45691446776415,
 'best_ml_price': 82.45691446776415,
 'best_ml_model': 'Approach 1 - Random Forest volatility forecast plus BSM',
 'approach2_direct_price': 0.0,
 'error_margin_90': [51.278806405868096, 94.1532512568125],
 'error_margin_note': 'This is a training-block out-of-fold proxy-error band evaluated on the later test period, not a chooser market-price confidence interval.'}


## 3. Out-of-Fold Error-Band Calibration and Test Coverage

In [3]:
pred = pd.read_csv(TOOL_DIR / "assets" / "data" / "test_predictions.csv", parse_dates=["Date"])
lower = pred["Approach1_MLVol_BSM_Price"] + ERROR_INTERVAL["lower_residual_quantile"]
upper = pred["Approach1_MLVol_BSM_Price"] + ERROR_INTERVAL["upper_residual_quantile"]
covered = pred["Target_Chooser_Proxy_Price"].between(lower, upper)
calibration = pd.DataFrame([{
    "Nominal_Coverage": ERROR_INTERVAL["nominal_coverage"],
    "Empirical_Test_Coverage": covered.mean(),
    "Mean_Interval_Width": (upper-lower).mean(),
    "Lower_Residual_Quantile": ERROR_INTERVAL["lower_residual_quantile"],
    "Upper_Residual_Quantile": ERROR_INTERVAL["upper_residual_quantile"],
    "Interpretation": ERROR_INTERVAL["limitation"],
}])
calibration.to_csv(RESULTS_DIR / "error_margin_calibration.csv", index=False)
metrics = pd.read_csv(TOOL_DIR / "assets" / "data" / "model_comparison.csv")
metrics.to_csv(RESULTS_DIR / "final_model_comparison.csv", index=False)
calibration

Out[3]: 
   Nominal_Coverage  ...                                     Interpretation
0               0.9  ...  This is a training-block out-of-fold proxy-err...

[1 rows x 6 columns]


## 4. Required stress scenarios

In [4]:
scenarios = []
for label, vol_mult, rate_add in [
    ("Base", 1.0, 0.0), ("50% Volatility Spike", 1.5, 0.0),
    ("2% Rate Hike", 1.0, 0.02), ("Combined Shock", 1.5, 0.02)
]:
    result = price_contract(
        snapshot["jpm_close"], 150.0, snapshot["risk_free_rate"] + rate_add,
        0.0233, snapshot["historical_volatility_20d"] * vol_mult,
        0.5, 1.0, snapshot["vix_close"] * vol_mult
    )
    scenarios.append({
        "Scenario": label, "Input_Rate": snapshot["risk_free_rate"] + rate_add,
        "BSM_Price": result["bsm_price"],
        "Approach1_Price": result["approach1_ml_vol_bsm_price"],
        "Best_ML_Price": result["best_ml_price"],
        "Approach2_Direct_Price": result["approach2_direct_price"],
    })
scenario_df = pd.DataFrame(scenarios)
for column in ["BSM_Price", "Approach1_Price", "Best_ML_Price"]:
    scenario_df[column + "_Change"] = scenario_df[column] - scenario_df.loc[0, column]
scenario_df.to_csv(RESULTS_DIR / "final_stress_scenarios.csv", index=False)
scenario_df

Out[4]: 
               Scenario  ...  Best_ML_Price_Change
0                  Base  ...              0.000000
1  50% Volatility Spike  ...              1.169501
2          2% Rate Hike  ...              2.834276
3        Combined Shock  ...              3.794094

[4 rows x 9 columns]


## 5. Integration checks

In [5]:
checks = []
def check(name, condition, detail):
    checks.append({"Check": name, "Passed": bool(condition), "Detail": str(detail)})

check("Standalone BSM finite", np.isfinite(example["bsm_price"]) and example["bsm_price"] > 0, example["bsm_price"])
check("Best ML finite", np.isfinite(example["best_ml_price"]) and example["best_ml_price"] >= 0, example["best_ml_price"])
check("Error band ordered", example["error_margin_90"][0] <= example["best_ml_price"] <= example["error_margin_90"][1], example["error_margin_90"])
check("Volatility spike increases BSM", scenario_df.loc[1,"BSM_Price"] > scenario_df.loc[0,"BSM_Price"], scenario_df.loc[1,"BSM_Price_Change"])
observed_rate_shock = scenario_df.loc[scenario_df["Scenario"] == "2% Rate Hike", "Input_Rate"].iloc[0] - scenario_df.loc[0, "Input_Rate"]
check("Required rate shock is 2 percentage points", abs(observed_rate_shock - 0.02) < 1e-12, observed_rate_shock)
check("ML contract restriction", not price_contract(snapshot["jpm_close"], strike=160.0)["ml_available"], "K=160 rejected for ML")
check("Test rows preserved", len(pred) == 252, len(pred))
check("All model metrics finite", metrics[["MAE","RMSE","R2"]].notna().all().all(), "MAE/RMSE/R2")
checks_df = pd.DataFrame(checks)
checks_df.to_csv(RESULTS_DIR / "integration_checks.csv", index=False)
assert checks_df["Passed"].all(), checks_df.loc[~checks_df["Passed"]]
checks_df

Out[5]: 
                                        Check  ...                                  Detail
0                       Standalone BSM finite  ...                       82.52782127278121
1                              Best ML finite  ...                       82.45691446776415
2                          Error band ordered  ...  [51.278806405868096, 94.1532512568125]
3              Volatility spike increases BSM  ...                      1.2726001155848365
4  Required rate shock is 2 percentage points  ...                    0.020000000000000004
5                     ML contract restriction  ...                   K=160 rejected for ML
6                         Test rows preserved  ...                                     252
7                    All model metrics finite  ...                             MAE/RMSE/R2

[8 rows x 3 columns]


## 6. Final figures

In [6]:
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.plot(pred["Date"], pred["Target_Chooser_Proxy_Price"], label="Forward-volatility proxy target", linewidth=2)
ax.plot(pred["Date"], pred["Current_Chooser_BSM_Price"], label="Week 4 BSM", alpha=.85)
ax.plot(pred["Date"], pred["Approach1_MLVol_BSM_Price"], label="Selected ML", alpha=.9)
ax.plot(pred["Date"], pred["Approach2_Direct_Price"], label="Direct-price benchmark", alpha=.75)
ax.fill_between(pred["Date"], lower, upper, alpha=.16, label="Empirical 90% proxy-error band")
ax.set(title="Final Test-Period Pricing Comparison", ylabel="Chooser price ($)")
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "final_pricing_comparison.png", dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
positions = np.arange(len(scenario_df))
width = .26
for offset, column, label in [(-width,"BSM_Price","BSM"),(0,"Best_ML_Price","Selected ML"),(width,"Approach2_Direct_Price","Direct-price benchmark")]:
    ax.bar(positions + offset, scenario_df[column], width, label=label)
ax.set_xticks(positions, scenario_df["Scenario"], rotation=12)
ax.set_ylabel("Chooser price ($)")
ax.set_title("Final Required Stress Scenarios")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "final_stress_scenarios.png", dpi=180)
plt.close(fig)
print("Saved final figures.")

Saved final figures.


## 7. Conclusion

The full Week 1–8 chain is preserved. The final tool provides BSM and ML pricing, an explicitly qualified proxy-error band, sensitivity/stress views, model-performance evidence, and key-free public-data refresh with a cached fallback. Hosting and the narrated demo remain student-account/personal actions.